# Feature Engineering
The process of creating, transforming, or selecting input variables (features) from raw data to improve the performance of machine learning models. It often involves techniques like scaling, encoding categorical data, creating new features from existing ones, or reducing dimensionality. Essentially, it’s about turning raw data into meaningful inputs that help models learn patterns more effectively.

Create meaningful features:
- One-hot encoding
- Extract date features (weekday, month, hour…)
- Domain logic features
- Combine or derive features (ratios, interactions)
- Transform skew (log transform)
- Text preprocessing (tokenisation, embeddings)

Here we will be using pandas to handle storing and processing the data

In [1]:
import pandas as pd

df = pd.read_csv("data/nba_games.csv",index_col=0)

In [2]:
df.head()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,ft%,...,usg%_max_opp,ortg_max_opp,drtg_max_opp,team_opp,total_opp,home_opp,season,date,won,target
2630,240.0,39.0,75.0,0.520,13.0,19.0,0.684,22.0,22.0,1.000,...,29.7,150.0,133.0,UTA,104,0,2017,2016-10-25,True,0
2631,240.0,40.0,82.0,0.488,8.0,24.0,0.333,16.0,16.0,1.000,...,39.2,220.0,120.0,POR,113,1,2017,2016-10-25,False,1
2632,240.0,40.0,85.0,0.471,7.0,33.0,0.212,13.0,18.0,0.722,...,36.2,263.0,112.0,SAS,129,0,2017,2016-10-25,False,1
2633,240.0,47.0,98.0,0.480,12.0,24.0,0.500,23.0,26.0,0.885,...,31.8,139.0,143.0,GSW,100,1,2017,2016-10-25,True,1
2634,240.0,45.0,94.0,0.479,13.0,35.0,0.371,14.0,19.0,0.737,...,36.8,153.0,125.0,NYK,88,0,2017,2016-10-25,True,1


In [3]:
def remove_cols(df, cols):
    for col in cols:
        del df[col]
    return df

## Rolling averages
First useful feature to add to our dataframe is the rolling averages which our model can use to see how each team is performing over their last 10 games

In [4]:
removed_columns = ["season", "date", "won", "target", "team", "team_opp"]
selected_columns = df.columns[~df.columns.isin(removed_columns)]

In [5]:
def find_team_averages(team):
    #rolling = team.rolling(10).mean()
    rolling = team.shift().rolling(10, min_periods=1).mean()
    return rolling

df_rolling = df[list(selected_columns) + ["won", "team", "season"]]
df_rolling = df_rolling.groupby(["team", "season"], group_keys=False).apply(find_team_averages)

C:\Users\henry\AppData\Local\Temp\ipykernel_14052\4101528943.py:3: FutureWarning: Dropping of nuisance columns in rolling operations is deprecated; in a future version this will raise TypeError. Select only valid columns before calling the operation. Dropped columns were Index(['team'], dtype='object')
  rolling = team.shift().rolling(10, min_periods=1).mean()


In [6]:
def find_team_averages(group):
    # Compute rolling mean with a 10-game window, shifted so current row isn’t included
    return group.shift().rolling(10, min_periods=1).mean()

# Select relevant columns
cols_to_roll = list(selected_columns) + ["won"]

# Apply rolling per team per season
df_rolling = df.groupby(["team", "season"], group_keys=False)[cols_to_roll].apply(find_team_averages)

In [7]:
df_rolling.head()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,ft%,...,ast%_max_opp,stl%_max_opp,blk%_max_opp,tov%_max_opp,usg%_max_opp,ortg_max_opp,drtg_max_opp,total_opp,home_opp,won
2630,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2631,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2632,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2633,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2634,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
rolling_cols = [f"{col}_10" for col in df_rolling.columns]
df_rolling.columns = rolling_cols

In [22]:
df_rolling.head()

,mp_10,fg_10,fga_10,fg%_10,3p_10,3pa_10,3p%_10,ft_10,fta_10,ft%_10,...,ast%_max_opp_10,stl%_max_opp_10,blk%_max_opp_10,tov%_max_opp_10,usg%_max_opp_10,ortg_max_opp_10,drtg_max_opp_10,total_opp_10,home_opp_10,won_10
2630,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2631,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2632,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2633,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2634,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df = pd.concat([df, df_rolling], axis=1)

In [10]:
df = df.dropna() #Drop all null/na values

In [11]:
def shift_col(team, col_name):
    next_col = team[col_name].shift(-1)
    return next_col

def add_col(df, col_name):
    return df.groupby("team", group_keys=False).apply(lambda x: shift_col(x, col_name))

df["home_next"] = add_col(df, "home")
df["team_opp_next"] = add_col(df, "team_opp")
df["date_next"] = add_col(df, "date")

df = df.copy()

In [12]:
full = df.merge(
    df[rolling_cols + ["team_opp_next", "date_next", "team"]], 
    left_on=["team", "date_next"], 
    right_on=["team_opp_next", "date_next"]
)

In [13]:
full["date"] = pd.to_datetime(full["date"])
full["date_next"] = pd.to_datetime(full["date_next"])

In [14]:
full["days_rested"] = (full["date_next"] - full["date"]).dt.days

In [15]:
full = full[full["days_rested"] <= 4]

# Final Dataframe
The final features must only include past data and current information, that can both be inputted into the model.
#### Past data:
- Rolling Averages for each basic and advanced stat
- Days Rested
#### Current data:
- Home team
- Away team
- Season
- Date

In [20]:
full.columns[full.dtypes == "object"]

Index(['team_x', 'team_opp', 'team_opp_next_x', 'team_opp_next_y', 'team_y'], dtype='object')

In [16]:
removed_columns = removed_columns + list(full.columns[full.dtypes == "object"])

In [17]:
selected_columns = full.columns[~full.columns.isin(removed_columns)] + list(full.columns['_10'])

In [ ]:
selected_columns = ["season", "date","team_x", "team_y"] + ["won"]

For the ai model to predict which team is going to win, it needs the recent stats from each team playing.

### The primary key used for finding the most recent game played and pull statistics from that game
- season
- date
- team_x (home team)
- team_y (away team)

### Information the model uses to make predictions
- rolling 10-game averages team x (contains '_10_x')
- rolling 10-game averages team y (contains '_10_y')
- rolling 5-game averages team x (contains '_5_x')
- rolling 5-game averages team y (contains '_5_y')
- team x performance against team y in past 3 games
- team y performance against team x in past 3 games
- days rested team x
- days rested team y

### Label/Outcome/Loss/Class
- won   <-- used as label, test model performance with this feature

In [1]:
for column in removed_columns:
    print(column)

NameError: name 'removed_columns' is not defined

In [21]:
for column in selected_columns:
    print(column)

mp
fg
fga
fg%
3p
3pa
3p%
ft
fta
ft%
orb
drb
trb
ast
stl
blk
tov
pf
pts
ts%
efg%
3par
ftr
orb%
drb%
trb%
ast%
stl%
blk%
tov%
usg%
ortg
drtg
fg_max
fga_max
fg%_max
3p_max
3pa_max
3p%_max
ft_max
fta_max
ft%_max
orb_max
drb_max
trb_max
ast_max
stl_max
blk_max
tov_max
pf_max
pts_max
gmsc_max
+/-_max
ts%_max
efg%_max
3par_max
ftr_max
orb%_max
drb%_max
trb%_max
ast%_max
stl%_max
blk%_max
tov%_max
usg%_max
ortg_max
drtg_max
total
home
mp_opp
fg_opp
fga_opp
fg%_opp
3p_opp
3pa_opp
3p%_opp
ft_opp
fta_opp
ft%_opp
orb_opp
drb_opp
trb_opp
ast_opp
stl_opp
blk_opp
tov_opp
pf_opp
pts_opp
ts%_opp
efg%_opp
3par_opp
ftr_opp
orb%_opp
drb%_opp
trb%_opp
ast%_opp
stl%_opp
blk%_opp
tov%_opp
usg%_opp
ortg_opp
drtg_opp
fg_max_opp
fga_max_opp
fg%_max_opp
3p_max_opp
3pa_max_opp
3p%_max_opp
ft_max_opp
fta_max_opp
ft%_max_opp
orb_max_opp
drb_max_opp
trb_max_opp
ast_max_opp
stl_max_opp
blk_max_opp
tov_max_opp
pf_max_opp
pts_max_opp
gmsc_max_opp
+/-_max_opp
ts%_max_opp
efg%_max_opp
3par_max_opp
ftr_max_opp
orb%_max_op

In [19]:
full.to_pickle('df_nba_games.pkl')